# Validation Integration with Versioning - Real Dataset Demo

This notebook demonstrates validation integration with dataset versioning using **real production data**.

**Dataset**: `notebooks/data/products.parquet` (make sure to have a copy of a dataset created with the pipeline)

**Manifest**: `notebooks/data/verification_manifest.json` (make sure to have a copy of a manifest created with the pipeline)

## Features Demonstrated
1. Creating multiple dataset versions
2. Validating current dataset (symlink)
3. Validating specific versions
4. Version-aware validation reports
5. Validating versions created with append mode
6. Comparing validation results across versions

In [ ]:
# Imports
from pathlib import Path
import os
import sys

import pandas as pd
from loguru import logger

# Add the project root to the python path
# ruff: noqa: E402
root_path = Path(os.getcwd()).parent
# ruff: noqa: E402
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.building.builder import DatasetBuilder
from src.storage.versioning import DatasetVersionManager
from src.validation.validator import DatasetValidator

# Configure logger for notebook
logger.remove()
logger.add(
    lambda msg: print(msg, end=""), colorize=True, format="<level>{message}</level>"
)

## Setup: Paths and Initial State

In [ ]:
# Paths (using notebooks/data/ directory)
dataset_path = Path("notebooks/data/products.parquet")
manifest_path = Path("notebooks/data/verification_manifest.json")
output_dir = dataset_path.parent

print(f"Dataset path: {dataset_path}")
print(f"Manifest path: {manifest_path}")
print(f"Output directory: {output_dir}")
print(f"Dataset exists: {dataset_path.exists()}")
print(f"Manifest exists: {manifest_path.exists()}")

## Step 1: Create Version 1

Build the first version of the dataset with auto-versioning.

In [ ]:
# Build dataset version 1
builder1 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    version_mode="auto",
    force=True,
)

print("Building dataset version 1...\n")
success = builder1.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

## Step 2: Validate Version 1

Validate the first version using version-aware validation.

In [ ]:
# Get version 1 info
version_manager = DatasetVersionManager(output_dir)
v1_info = version_manager.get_version_info(1)

print("📝 Version 1 Metadata:")
print(f"  Version: {v1_info['version']}")
print(f"  Timestamp: {v1_info['timestamp']}")
print(f"  Records: {v1_info['record_count']}")
print(f"  Manifest: {v1_info['manifest_file']}")
print(f"  Manifest Hash: {v1_info['manifest_hash'][:32]}...")
print(f"  Append Mode: {v1_info['append_mode']}")

In [ ]:
# Validate version 1 with version info
versioned_path_v1 = version_manager.get_version_path(1)
versioned_manifest_v1 = output_dir / v1_info["manifest_file"]

validator_v1 = DatasetValidator(
    parquet_path=versioned_path_v1,
    manifest_path=versioned_manifest_v1,
    version_number=1,
    version_info=v1_info,
)

print("\n🔍 Validating Version 1...\n")
success_v1 = validator_v1.validate_all(verbose=False, project_root=Path.cwd())

## Step 3: Create Version 2 with Append

Create a second version using append mode to demonstrate version-aware validation with append metadata.

In [ ]:
# Build version 2 with append
builder2 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    append=True,
    merge_strategy="update",
    version_mode="auto",
    force=True,
)

print("Building dataset version 2 with append...\n")
success = builder2.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

## Step 4: Validate Version 2

Validate version 2 and observe the append mode information in the validation report.

In [ ]:
# Get version 2 info
v2_info = version_manager.get_version_info(2)

print("📝 Version 2 Metadata:")
print(f"  Version: {v2_info['version']}")
print(f"  Timestamp: {v2_info['timestamp']}")
print(f"  Records: {v2_info['record_count']}")
print(f"  Manifest: {v2_info['manifest_file']}")
print(f"  Append Mode: {v2_info['append_mode']}")
print(f"  Parent Version: {v2_info.get('parent_version', 'N/A')}")
print(f"  Records Added: {v2_info.get('records_added', 0)}")
print(f"  Records Updated: {v2_info.get('records_updated', 0)}")

In [ ]:
# Validate version 2 with version info
versioned_path_v2 = version_manager.get_version_path(2)
versioned_manifest_v2 = output_dir / v2_info["manifest_file"]

validator_v2 = DatasetValidator(
    parquet_path=versioned_path_v2,
    manifest_path=versioned_manifest_v2,
    version_number=2,
    version_info=v2_info,
)

print("\n🔍 Validating Version 2...\n")
success_v2 = validator_v2.validate_all(verbose=False, project_root=Path.cwd())

## Step 5: Create Version 3 (Overwrite Mode)

Create version 3 without append mode to show the difference in validation reports.

In [ ]:
# Build version 3 without append
builder3 = DatasetBuilder(
    manifest_path=manifest_path,
    output_path=dataset_path,
    append=False,
    version_mode="auto",
    force=True,
)

print("Building dataset version 3 (overwrite mode)...\n")
success = builder3.build_and_save()
print(f"\n{'✓' if success else '✗'} Build result: {'SUCCESS' if success else 'FAILED'}")

In [ ]:
# Get and display version 3 info
v3_info = version_manager.get_version_info(3)

print("📝 Version 3 Metadata:")
print(f"  Version: {v3_info['version']}")
print(f"  Timestamp: {v3_info['timestamp']}")
print(f"  Records: {v3_info['record_count']}")
print(f"  Append Mode: {v3_info['append_mode']}")
print(f"  Parent Version: {v3_info.get('parent_version', 'None (overwrite mode)')}")

## Step 6: Validate Current Dataset (Symlink)

Validate the current dataset using the symlink. This demonstrates backward compatibility - validation works without version info.

In [ ]:
# Validate current dataset (symlink) without version info
validator_current = DatasetValidator(
    parquet_path=dataset_path,
    manifest_path=manifest_path,
)

print("🔍 Validating Current Dataset (symlink, no version info)...\n")
success_current = validator_current.validate_all(verbose=False, project_root=Path.cwd())

print(
    "\n📌 Note: Current dataset validation works without version info (backward compatible)"
)

## Step 7: List All Versions

View complete version history.

In [ ]:
# List all versions
versions = version_manager.list_versions()

print(f"📊 Dataset Version History ({len(versions)} versions)\n")

for v in versions:
    print(f"Version {v['version']}:")
    print(f"  Timestamp: {v['timestamp']}")
    print(f"  Records: {v['record_count']}")
    print(f"  File: {v['file']}")
    print(f"  Manifest: {v['manifest_file']} ({v['manifest_hash'][:16]}...)")
    print(f"  Append Mode: {v.get('append_mode', False)}")

    if v.get("append_mode"):
        print(
            f"  Added: {v.get('records_added', 0)}, Updated: {v.get('records_updated', 0)}"
        )
        print(f"  Parent: v{v.get('parent_version', 'N/A')}")
    print()

print(f"📌 Current Version: {version_manager.get_current_version()}")

## Step 8: Validate Older Version

Demonstrate validating an older version (version 1) after newer versions have been created.

In [ ]:
# Re-validate version 1 (older version)
print("🔍 Re-validating Version 1 (older version)...\n")

validator_v1_again = DatasetValidator(
    parquet_path=versioned_path_v1,
    manifest_path=versioned_manifest_v1,
    version_number=1,
    version_info=v1_info,
)

success_v1_again = validator_v1_again.validate_all(
    verbose=False, project_root=Path.cwd()
)

print("\n✓ Older versions remain accessible and can be validated at any time")

## Step 9: Compare Validation Results Across Versions

Extract and compare key metrics from validation results.

In [ ]:
# Compare validation stats across versions
print("📊 Validation Comparison Across Versions\n")
print(f"{'Version':<10} {'Records':<10} {'Errors':<10} {'Warnings':<10} {'Status':<10}")
print("-" * 50)

# Version 1
print(
    f"{'v1':<10} {validator_v1.stats.get('total_records', 0):<10} {len(validator_v1.errors):<10} {len(validator_v1.warnings):<10} {'✓ PASS' if success_v1 else '✗ FAIL':<10}"
)

# Version 2
print(
    f"{'v2':<10} {validator_v2.stats.get('total_records', 0):<10} {len(validator_v2.errors):<10} {len(validator_v2.warnings):<10} {'✓ PASS' if success_v2 else '✗ FAIL':<10}"
)

# Current (symlink)
print(
    f"{'current':<10} {validator_current.stats.get('total_records', 0):<10} {len(validator_current.errors):<10} {len(validator_current.warnings):<10} {'✓ PASS' if success_current else '✗ FAIL':<10}"
)

print("\n📈 Coverage Metrics:")
print("\nVersion 1:")
print(f"  Images: {validator_v1.stats.get('image_coverage', 'N/A')}")
print(f"  PDFs: {validator_v1.stats.get('pdf_coverage', 'N/A')}")
print(f"  Specs: {validator_v1.stats.get('specs_coverage', 'N/A')}")

print("\nVersion 2:")
print(f"  Images: {validator_v2.stats.get('image_coverage', 'N/A')}")
print(f"  PDFs: {validator_v2.stats.get('pdf_coverage', 'N/A')}")
print(f"  Specs: {validator_v2.stats.get('specs_coverage', 'N/A')}")

## Step 10: Verify Symlinks Point to Current Version

In [ ]:
# Check symlinks
dataset_symlink = output_dir / "products.parquet"
manifest_symlink = output_dir / "verification_manifest.json"

current_version = version_manager.get_current_version()

print("🔗 Symlink Status:")
print(f"  products.parquet exists: {dataset_symlink.exists()}")
print(f"  verification_manifest.json exists: {manifest_symlink.exists()}")
print(f"\n  Current version: v{current_version}")

# Verify symlinks point to current version
if dataset_symlink.exists():
    df_symlink = pd.read_parquet(dataset_symlink)
    df_current = pd.read_parquet(output_dir / f"products_v{current_version}.parquet")
    print(f"  Symlink points to current version: {len(df_symlink) == len(df_current)}")
    print(f"  Symlink records: {len(df_symlink)}")
    print(f"  Version {current_version} records: {len(df_current)}")

## Step 11: Cleanup Test Files

Remove all versioned files and metadata to reset for future runs.

In [ ]:
# List all version-related files
print("🗑️  Cleaning up test files...")

files_to_remove = [
    "dataset_metadata.json",
    "products.parquet",  # symlink
    "verification_manifest.json",  # symlink
]

# Add versioned files
for i in range(1, 10):  # Check up to v10
    files_to_remove.append(f"products_v{i}.parquet")
    files_to_remove.append(f"verification_manifest_v{i}.json")

removed_count = 0
for filename in files_to_remove:
    file_path = output_dir / filename
    if file_path.exists():
        file_path.unlink()
        removed_count += 1
        print(f"  ✓ Removed: {filename}")

print(f"\n✓ Cleanup complete: {removed_count} files removed")

## Summary

This notebook demonstrated:

1. ✅ **Version-aware validation** - Validating specific dataset versions with metadata
2. ✅ **Version information in reports** - Timestamps, record counts, append mode details
3. ✅ **Backward compatibility** - Validation works without version info (symlink validation)
4. ✅ **Append mode tracking** - Version reports show parent version and merge statistics
5. ✅ **Historical validation** - Older versions remain accessible and can be validated
6. ✅ **Validation comparison** - Compare metrics across different versions

### Key Features

**Version-Aware Validation:**
```python
# Validate specific version
version_manager = DatasetVersionManager(output_dir)
version_info = version_manager.get_version_info(2)

validator = DatasetValidator(
    parquet_path=versioned_path,
    manifest_path=versioned_manifest,
    version_number=2,
    version_info=version_info,
)

validator.validate_all()
```

**Validation Report Includes:**
- Version number
- Timestamp (ISO 8601 with timezone)
- Record count
- Manifest hash (SHA-256)
- Append mode flag
- Records added/updated (if append mode)
- Parent version (if append mode)

### CLI Quick Reference

```bash
# Validate current dataset (symlink)
python validate_dataset.py

# Validate specific version
python validate_dataset.py --version 2

# Validate version with verbose output
python validate_dataset.py --version 1 --verbose

# List all available versions
python build_dataset.py --list-versions
```

### Integration with Phases 1 & 2

**Complete Workflow:**
```bash
# Phase 1: Build with versioning
python build_dataset.py --version-mode auto

# Phase 2: Append and create new version
python build_dataset.py --append --version-mode auto

# Phase 3: Validate specific version
python validate_dataset.py --version 2

# Phase 3: Validate current (latest) version
python validate_dataset.py
```